In [ ]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Original dataset shape:", df.shape)
display(df.head())

Saving movies_feature_engineered.csv to movies_feature_engineered (2).csv
Original dataset shape: (4998, 31)


,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes,production_company,release_month,marketing_budget
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000,Sun Pictures,2,9835844
1,Color,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,PG-13,300000000.0,2007.0,5000.0,7.1,2.35,0,Lyca Productions,10,77621649
2,Color,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,PG-13,245000000.0,2015.0,393.0,6.8,2.35,85000,Red Giant Movies,8,65802580
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,PG-13,250000000.0,2012.0,23000.0,8.5,2.35,164000,AGS Entertainment,6,44448965
4,Color,Doug Walker,110.0,103.0,131.0,371.5,Rob Walker,131.0,25517500.0,Documentary,...,R,20000000.0,2005.0,12.0,7.1,2.35,0,Sree Gokulam Movies,6,43868508


In [ ]:
if "imdb_score" not in df.columns:
    raise ValueError("The dataset must contain an imdb_score column.")

# Create the classification target
df["success"] = (df["imdb_score"] >= 7.0).astype(int)

# Keep IMDb score for analysis, but not as a predictor
excluded_cols = ["imdb_score", "success", "movie_imdb_link"]

X = df.drop(columns=excluded_cols, errors="ignore").copy()
y = df["success"].copy()

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))
print("Target distribution:")
print(y.value_counts())

Numerical features: 17
Categorical features: 12
Target distribution:
success
0    3240
1    1758
Name: count, dtype: int64


In [ ]:
# Handle missing numerical values
if numeric_cols:
    num_imputer = SimpleImputer(strategy="median")
    X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])

# Handle missing categorical values and frequency-encode
for col in categorical_cols:
    X[col] = X[col].fillna("Unknown").astype(str)

    frequencies = X[col].value_counts(normalize=True)
    X[col] = X[col].map(frequencies)

print("Columns after frequency encoding:", X.shape[1])

Columns after frequency encoding: 29


In [ ]:
skewed_cols = []

for col in numeric_cols:
    if X[col].nunique() > 2 and X[col].min() >= 0:
        if abs(X[col].skew()) > 1:
            X[col] = np.log1p(X[col])
            skewed_cols.append(col)

print("Columns corrected for skewness:", skewed_cols)
print("Columns after skewness correction:", X.shape[1])

Columns corrected for skewness: ['num_critic_for_reviews', 'duration', 'director_facebook_likes', 'actor_3_facebook_likes', 'actor_1_facebook_likes', 'gross', 'num_voted_users', 'cast_total_facebook_likes', 'facenumber_in_poster', 'num_user_for_reviews', 'budget', 'title_year', 'actor_2_facebook_likes', 'aspect_ratio', 'movie_facebook_likes']
Columns after skewness correction: 29


In [ ]:
scaler = MinMaxScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Columns after scaling:", X_scaled.shape[1])
display(X_scaled.head())

Columns after scaling: 29


,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,country,content_rating,budget,title_year,actor_2_facebook_likes,aspect_ratio,movie_facebook_likes,production_company,release_month,marketing_budget
0,1.0,0.046875,0.980500,0.747303,0.000000,0.672316,0.09375,0.516766,1.000000,0.051282,...,1.000000,0.601751,0.778975,0.931631,0.578528,0.118374,0.815203,1.0,0.090909,0.088799
1,1.0,0.046875,0.835537,0.734898,0.630775,0.687897,0.18750,0.792616,0.941433,0.123932,...,1.000000,0.601751,0.792190,0.912054,0.720120,0.209181,0.000000,1.0,0.818182,0.774020
2,1.0,0.054688,0.950066,0.703195,0.000000,0.506567,0.03125,0.696057,0.913043,0.192308,...,0.117024,0.601751,0.780836,0.990248,0.505282,0.209181,0.889334,1.0,0.636364,0.654546
3,1.0,0.054688,1.000000,0.727720,0.995574,1.000000,0.12500,0.763218,0.965557,0.123932,...,1.000000,0.601751,0.781969,0.960961,0.849130,0.209181,0.940828,1.0,0.454545,0.438690
4,1.0,0.000000,0.668415,0.616740,0.486175,0.589472,0.00000,0.365227,0.778935,0.213675,...,1.000000,1.000000,0.640368,0.892456,0.216859,0.209181,0.000000,1.0,0.454545,0.432823


In [ ]:
k = min(30, X_scaled.shape[1])

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=k
)

X_selected_array = selector.fit_transform(X_scaled, y)

selected_cols = X_scaled.columns[selector.get_support()].tolist()

X_selected = pd.DataFrame(
    X_selected_array,
    columns=selected_cols,
    index=df.index
)

print("Selected predictor columns:", len(selected_cols))
print("Selected features:")
print(selected_cols)

Selected predictor columns: 29
Selected features:
['color', 'director_name', 'num_critic_for_reviews', 'duration', 'director_facebook_likes', 'actor_3_facebook_likes', 'actor_2_name', 'actor_1_facebook_likes', 'gross', 'genres', 'actor_1_name', 'movie_title', 'num_voted_users', 'cast_total_facebook_likes', 'actor_3_name', 'facenumber_in_poster', 'plot_keywords', 'num_user_for_reviews', 'language', 'country', 'content_rating', 'budget', 'title_year', 'actor_2_facebook_likes', 'aspect_ratio', 'movie_facebook_likes', 'production_company', 'release_month', 'marketing_budget']


In [ ]:
processed_df = X_selected.copy()

processed_df["imdb_score"] = df["imdb_score"].values
processed_df["success"] = y.values

print("Final dataset shape:", processed_df.shape)
print("Final number of columns:", processed_df.shape[1])
print("Missing values:", processed_df.isnull().sum().sum())

display(processed_df.head())

Final dataset shape: (4998, 31)
Final number of columns: 31
Missing values: 0


,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,budget,title_year,actor_2_facebook_likes,aspect_ratio,movie_facebook_likes,production_company,release_month,marketing_budget,imdb_score,success
0,1.0,0.046875,0.980500,0.747303,0.000000,0.672316,0.09375,0.516766,1.000000,0.051282,...,0.778975,0.931631,0.578528,0.118374,0.815203,1.0,0.090909,0.088799,7.9,1
1,1.0,0.046875,0.835537,0.734898,0.630775,0.687897,0.18750,0.792616,0.941433,0.123932,...,0.792190,0.912054,0.720120,0.209181,0.000000,1.0,0.818182,0.774020,7.1,1
2,1.0,0.054688,0.950066,0.703195,0.000000,0.506567,0.03125,0.696057,0.913043,0.192308,...,0.780836,0.990248,0.505282,0.209181,0.889334,1.0,0.636364,0.654546,6.8,0
3,1.0,0.054688,1.000000,0.727720,0.995574,1.000000,0.12500,0.763218,0.965557,0.123932,...,0.781969,0.960961,0.849130,0.209181,0.940828,1.0,0.454545,0.438690,8.5,1
4,1.0,0.000000,0.668415,0.616740,0.486175,0.589472,0.00000,0.365227,0.778935,0.213675,...,0.640368,0.892456,0.216859,0.209181,0.000000,1.0,0.454545,0.432823,7.1,1


In [ ]:
output_file = "movies_success_feature_engineered.csv"

processed_df.to_csv(output_file, index=False)

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>